# 10. Feature Selection

This notebook prepares the engineered Telco churn dataset for feature selection.

Goals:
- Load the engineered dataset
- Split features and target into train/test sets
- Check feature quality before selection
- Prepare a clean numeric matrix for ranking methods
- Scale only the features needed for VIF or logistic-regression-based selection
- Group features for interpretation
- Export summary tables for reporting

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import RANDOM_STATE, TARGET_COLUMN, TEST_SIZE

sns.set_theme(style="whitegrid")

## Load Engineered Dataset

In [ ]:
target_column = TARGET_COLUMN
engineered_data_path = project_root / "data" / "processed" / "telco_churn_engineered.csv"

if not engineered_data_path.exists():
    raise FileNotFoundError(
        "Engineered dataset not found. Run notebook 09_feature_engineering.ipynb first "
        f"to create: {engineered_data_path}"
    )

df = pd.read_csv(engineered_data_path)

dataset_summary = {
    "dataset_source": str(engineered_data_path.relative_to(project_root)),
    "shape": df.shape,
    "target_column": target_column,
    "target_present": target_column in df.columns,
}

dataset_summary

In [ ]:
if target_column not in df.columns:
    raise KeyError(f"Target column '{target_column}' was not found in the loaded dataset.")

df.head()

## Train-Test Split

In [ ]:
X = df.drop(columns=[target_column])
y = df[target_column].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = {
    "X_shape": X.shape,
    "y_shape": y.shape,
    "X_train_shape": X_train.shape,
    "X_test_shape": X_test.shape,
    "y_train_shape": y_train.shape,
    "y_test_shape": y_test.shape,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
}

split_summary

## Data Quality Checks Before Selection

In [ ]:
missing_value_summary = X_train.isna().sum().sort_values(ascending=False)
missing_features = missing_value_summary[missing_value_summary > 0]

feature_variance = X_train.var(numeric_only=True)
constant_features = feature_variance[feature_variance == 0].index.tolist()
near_constant_threshold = 0.01
near_constant_features = feature_variance[
    (feature_variance > 0) & (feature_variance <= near_constant_threshold)
].sort_values().index.tolist()

non_numeric_features = X_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()

data_quality_summary = {
    "missing_feature_count": int(missing_features.shape[0]),
    "total_missing_values": int(missing_features.sum()) if not missing_features.empty else 0,
    "constant_feature_count": len(constant_features),
    "near_constant_feature_count": len(near_constant_features),
    "near_constant_threshold": near_constant_threshold,
    "non_numeric_feature_count": len(non_numeric_features),
}

data_quality_summary

In [ ]:
if constant_features:
    X_train = X_train.drop(columns=constant_features)
    X_test = X_test.drop(columns=constant_features)

encoding_check_df = pd.DataFrame(
    {
        "dtype": X_train.dtypes.astype(str),
        "unique_values": X_train.nunique(),
    }
).sort_index()

quality_actions = {
    "removed_constant_features": constant_features,
    "flagged_near_constant_features": near_constant_features[:15],
    "remaining_feature_count": X_train.shape[1],
    "non_numeric_features": non_numeric_features,
}

quality_actions

## Prepare Data for Selection Methods

We create:
- `X_train_ranking`, `X_test_ranking` for general ranking methods
- `X_train_scaled_for_selection`, `X_test_scaled_for_selection` for VIF and logistic-regression-based selection

Binary features remain unchanged, while non-binary numeric features are standardized using training-set statistics only.

In [ ]:
numeric_feature_columns = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()

X_train_numeric = X_train[numeric_feature_columns].astype(float).copy()
X_test_numeric = X_test[numeric_feature_columns].astype(float).copy()

# Use the clean numeric matrices for ranking methods that only require numeric inputs.
X_train_ranking = X_train_numeric.copy()
X_test_ranking = X_test_numeric.copy()

binary_like_features = [
    column
    for column in numeric_feature_columns
    if set(X_train_numeric[column].dropna().unique()).issubset({0.0, 1.0})
]
scale_sensitive_features = [
    column for column in numeric_feature_columns if column not in binary_like_features
]

X_train_scaled_for_selection = X_train_numeric.copy()
X_test_scaled_for_selection = X_test_numeric.copy()

scaling_parameters = {}
for column in scale_sensitive_features:
    train_mean = X_train_numeric[column].mean()
    train_std = X_train_numeric[column].std(ddof=0)
    if train_std == 0:
        X_train_scaled_for_selection[column] = 0.0
        X_test_scaled_for_selection[column] = 0.0
        scaling_parameters[column] = {"mean": float(train_mean), "std": 0.0}
        continue

    X_train_scaled_for_selection[column] = (X_train_numeric[column] - train_mean) / train_std
    X_test_scaled_for_selection[column] = (X_test_numeric[column] - train_mean) / train_std
    scaling_parameters[column] = {"mean": float(train_mean), "std": float(train_std)}

selection_prep_summary = {
    "numeric_feature_count": len(numeric_feature_columns),
    "binary_like_feature_count": len(binary_like_features),
    "scaled_feature_count": len(scale_sensitive_features),
    "train_ranking_shape": X_train_ranking.shape,
    "test_ranking_shape": X_test_ranking.shape,
    "train_numeric_shape": X_train_numeric.shape,
    "test_numeric_shape": X_test_numeric.shape,
    "train_scaled_shape": X_train_scaled_for_selection.shape,
    "test_scaled_shape": X_test_scaled_for_selection.shape,
}

selection_prep_summary

In [ ]:
selection_matrix_preview = pd.DataFrame(
    {
        "feature": numeric_feature_columns,
        "is_binary_like": [feature in binary_like_features for feature in numeric_feature_columns],
        "used_in_numeric_matrix": True,
        "used_in_ranking_matrix": True,
        "scaled_for_vif_logistic": [feature in scale_sensitive_features for feature in numeric_feature_columns],
    }
).sort_values(["scaled_for_vif_logistic", "feature"], ascending=[False, True]).reset_index(drop=True)

selection_matrix_preview.head(15)

## Feature Grouping

In [ ]:
current_features = X_train.columns.tolist()

base_numerical_features = [
    feature for feature in ["tenure", "MonthlyCharges", "TotalCharges"]
    if feature in current_features
]

base_binary_features = [
    feature
    for feature in [
        "gender",
        "SeniorCitizen",
        "Partner",
        "Dependents",
        "PhoneService",
        "Contract",
        "PaperlessBilling",
    ]
    if feature in current_features
]

engineered_features = [
    feature
    for feature in current_features
    if (
        feature.endswith("_log")
        or feature.endswith("_scaled")
        or "_x_" in feature
        or feature in {
            "avg_monthly_spend",
            "is_new_customer",
            "has_support_services",
            "is_month_to_month",
            "is_auto_payment",
            "service_count",
        }
    )
]

assigned_features = set(base_numerical_features) | set(base_binary_features) | set(engineered_features)
one_hot_encoded_features = [
    feature for feature in current_features if feature not in assigned_features
]

binary_features = sorted(base_binary_features)
numerical_features = sorted(base_numerical_features)
engineered_features = sorted(engineered_features)
one_hot_encoded_features = sorted(one_hot_encoded_features)

all_grouped_features = (
    numerical_features
    + binary_features
    + one_hot_encoded_features
    + engineered_features
)
ungrouped_features = sorted(set(current_features) - set(all_grouped_features))
duplicate_group_assignments = len(all_grouped_features) - len(set(all_grouped_features))

feature_group_summary = {
    "numerical_count": len(numerical_features),
    "binary_count": len(binary_features),
    "one_hot_encoded_count": len(one_hot_encoded_features),
    "engineered_count": len(engineered_features),
    "total_grouped_features": len(all_grouped_features),
    "current_feature_count": len(current_features),
    "ungrouped_feature_count": len(ungrouped_features),
    "duplicate_group_assignments": duplicate_group_assignments,
}

feature_group_summary

In [ ]:
if ungrouped_features:
    raise ValueError(f"Some features were not assigned to a group: {ungrouped_features}")

if duplicate_group_assignments:
    raise ValueError("Some features were assigned to more than one group.")

feature_group_map = {}
for feature in numerical_features:
    feature_group_map[feature] = "numerical"
for feature in binary_features:
    feature_group_map[feature] = "binary"
for feature in one_hot_encoded_features:
    feature_group_map[feature] = "one_hot_encoded_categorical"
for feature in engineered_features:
    feature_group_map[feature] = "engineered"

feature_group_df = pd.DataFrame(
    {
        "feature": current_features,
        "feature_group": [feature_group_map[feature] for feature in current_features],
    }
).sort_values(["feature_group", "feature"]).reset_index(drop=True)

feature_group_df.head(20)

## Export Feature Selection Prep Tables

In [ ]:
feature_selection_tables_dir = project_root / "reports" / "tables" / "feature_selection"
feature_selection_tables_dir.mkdir(parents=True, exist_ok=True)

data_quality_summary_df = pd.DataFrame([data_quality_summary])
quality_actions_df = pd.DataFrame(
    {
        "removed_constant_features": pd.Series(constant_features, dtype="object"),
        "flagged_near_constant_features": pd.Series(near_constant_features, dtype="object"),
    }
)
selection_prep_summary_df = pd.DataFrame([selection_prep_summary])
selection_matrix_preview_df = selection_matrix_preview.copy()
scaling_parameters_df = pd.DataFrame(scaling_parameters).T.rename_axis("feature").reset_index()
feature_group_summary_df = pd.DataFrame([feature_group_summary])

data_quality_summary_df.to_csv(feature_selection_tables_dir / "data_quality_summary.csv", index=False)
quality_actions_df.to_csv(feature_selection_tables_dir / "data_quality_actions.csv", index=False)
selection_prep_summary_df.to_csv(feature_selection_tables_dir / "selection_prep_summary.csv", index=False)
selection_matrix_preview_df.to_csv(feature_selection_tables_dir / "selection_matrix_preview.csv", index=False)
scaling_parameters_df.to_csv(feature_selection_tables_dir / "selection_scaling_parameters.csv", index=False)
feature_group_summary_df.to_csv(feature_selection_tables_dir / "feature_group_summary.csv", index=False)
feature_group_df.to_csv(feature_selection_tables_dir / "feature_group_mapping.csv", index=False)

exported_tables_summary = {
    "output_dir": str(feature_selection_tables_dir.relative_to(project_root)),
    "files": [
        "data_quality_summary.csv",
        "data_quality_actions.csv",
        "selection_prep_summary.csv",
        "selection_matrix_preview.csv",
        "selection_scaling_parameters.csv",
        "feature_group_summary.csv",
        "feature_group_mapping.csv",
    ],
}

exported_tables_summary